# MaxCut

The cost function for the MaxCut problem as defined in the [original QAOA paper](https://arxiv.org/abs/1411.4028) is

$$
    \hat C = \frac 12 \sum_{(i, j) \in E(G)} (1 - \hat Z_i \hat Z_j),
$$

where $E(G)$ is the set of edges of the graph $G$.

In [ ]:
using QAOA, LinearAlgebra
import Random, Distributions
import MFAOA

using PyPlot
# PyPlot.plt.style.use("paper.mplstyle")
using PyCall
nx = pyimport("networkx");

## QAOA

__Defining the problem by hand:__

In [ ]:
N = 4
graph = nx.cycle_graph(N) 

figure(figsize=(3, 2))
nx.draw(graph, with_labels=true)

Note that we have to __shift the edges by 1__ when going from Python to Julia:

In [ ]:
h = zeros(N)
J = zeros(N, N)
for edge in graph.edges
    J[(edge .+ (1, 1))...] = -1/2.
end
J = J + transpose(J)

In [ ]:
p = 1
max_cut_problem = QAOA.Problem(p, h, J)

__Using the wrapper function:__

In [ ]:
max_cut_problem = QAOA.max_cut(N, [edge .+ (1, 1) for edge in graph.edges], num_layers=p)

__Gradient optimization with [Zygote](https://fluxml.ai/Zygote.jl/latest/):__

In [ ]:
learning_rate = 0.01
cost, params, probs = QAOA.optimize_parameters(max_cut_problem, vcat([0.5 for _ in 1:p], [0.5 for _ in 1:p]); learning_rate=learning_rate)

__Optimization with [NLopt](https://nlopt.readthedocs.io/en/latest/):__

In [ ]:
cost, params, probs = QAOA.optimize_parameters(max_cut_problem, vcat([0.5 for _ in 1:p], [0.5 for _ in 1:p]), :LN_COBYLA)

In [ ]:
xlabels = []
for bstr in digits.(0:2^N-1, base=2, pad=N)
    push!(xlabels, "\$|" * prod([string(b) for b in bstr]) * "\\rangle\$")
end

figure(figsize=(5, 3.2))
ax = subplot(111)
bar(0:2^N-1, probs)
ax.set_xticks(0:2^N-1)
ax.set_xticklabels(xlabels, rotation=90)
minorticks_off()
tight_layout()

The states `5` $\equiv |1010\rangle$ and `10` $\equiv |0101\rangle$ are indeed the correct solutions!

## Mean-Field AOA with MFAOA.jl

The [mean-field approximate optimization algorithm](https://doi.org/10.1103/PRXQuantum.4.030335) (mean-field AOA) replaces every qubit by a classical spin vector $\mathbf{n}_i = (n_i^x, n_i^y, n_i^z)$ and runs a QAOA-like schedule on these spins. The package [MFAOA.jl](https://github.com/Quicopt/MFAOA.jl) implements it.

The MaxCut cost function has no local fields, so flipping every spin leaves it unchanged. The mean-field AOA cannot break this symmetry by itself, so we fix the last spin to $\sigma_N = +1$. Its couplings to the other spins then act on them as local fields $h_i = J_{iN}$:

In [ ]:
h_fixed = J[1:N-1, N]        # local fields from the fixed spin σ_N = +1
J_fixed = J[1:N-1, 1:N-1]    # couplings among the remaining spins

# QAOA.jl maximizes the cost function, MFAOA.jl minimizes by default
mf_problem = MFAOA.IsingProblem(h_fixed, J_fixed; sense = :max)

__Following the spins through the schedule:__ we take MFAOA's linear schedule with $p = 1000$ steps and record every spin vector after each step.

In [ ]:
p = 1000
mf_schedule = MFAOA.linear_schedule(p, 0.5)

workspace = MFAOA.Workspace(mf_problem)
spins = workspace.state                 # spins.x[i], spins.y[i], spins.z[i] are the components of n_i
history = [[spins.x spins.y spins.z]]   # history[k][i, :] is n_i after k - 1 steps
for k in 1:p
    MFAOA.step!(workspace, mf_problem, mf_schedule.β[k], mf_schedule.γ[k])
    push!(history, [spins.x spins.y spins.z])
end

In [ ]:
# plot x, y, and z of all spins that are not fixed
times = range(0, 1, p + 1)

figure(figsize=((N - 1) * 2.2, 2))
for i in 1:N-1
    subplot(1, N - 1, i)
    for (component, label) in enumerate(["x", "y", "z"])
        plot(times, [n[i, component] for n in history], label=label)
    end
    xlim(0, 1)
    ylim(-1, 1)
    xlabel("t/T")
    ylabel("n_" * string(i))
    i == 1 && legend(fontsize=7)
end
tight_layout()

__Reading off the solution:__ the sign of $n_i^z$ at the end of the schedule is the spin $\sigma_i$. `MFAOA.solve` runs the same schedule and returns these spins:

In [ ]:
result = MFAOA.solve(mf_problem, mf_schedule)
σ = vcat(result.σ, 1)       # append the fixed spin σ_N = +1
bits = (1 .- σ) .÷ 2        # a qubit in state |0⟩ has σ = +1
state = sum(bits[i] * 2^(i - 1) for i in 1:N)

println("solution |", join(bits), "⟩ = state ", state)
println("cut edges: ", count(σ[i] != σ[j] for j in 1:N for i in 1:j-1 if J[i, j] != 0), " of ", length(graph.edges))

This is the state `5` $\equiv |1010\rangle$ that the QAOA found above, and it cuts all four edges. The other solution, `10` $\equiv |0101\rangle$, has every spin flipped, so fixing $\sigma_4 = +1$ excludes it.

## Annealing

In [ ]:
T_anneal = 8.
p = 256
linear_schedule(t) = t / T_anneal
annealing_problem = QAOA.Problem(p, zeros(N), J);

In [ ]:
probs = anneal(annealing_problem, linear_schedule, T_anneal)

### Second-order schedule

In [ ]:
τ = T_anneal / p
γ = τ .* ((1:p) .- 1/2) ./ p |> collect
β = τ .* (1 .- (1:p) ./ p) |> collect
β[p] = τ / (4 * p);

In [ ]:
probs = anneal(annealing_problem, β, γ)